# NFL Player Contact Detection: YOLO and Video Feature Probe

This research notebook checks whether video-derived features can add signal to
the current tracking-first pipeline. It compares the competition-provided helmet
boxes with a lightweight YOLOv8 frame probe.

The key question is practical: do generic YOLO detections add useful body-level
context, or are the provided helmet boxes already the more reliable source for
player-specific video features?

## 1. Setup and Configuration

This is an EDA/research notebook, not a final submission notebook. Internet can
be enabled here so `ultralytics` and `yolov8n.pt` can be installed/downloaded.
A scored Kaggle submission must later use offline inputs for any package or
weight file it depends on.

In [ ]:
import importlib.util
import subprocess
import sys
import urllib.request
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython.display import Image, display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)

NOTEBOOK_VERSION = "YOLO_VIDEO_PROBE_V5_CLEAN"
NOTEBOOK_NAME = "YOLO and Video Feature Probe"
TARGET = "contact"
ID_COL = "contact_id"
VIDEO_FPS = 59.94
TRACKING_HZ = 10
SNAP_FRAME = int(round(VIDEO_FPS * 5))
RANDOM_STATE = 42
SAMPLE_ROWS = 50_000
YOLO_CONFIDENCE = 0.25
YOLO_RESEARCH_MODE = True
YOLO_MODEL_NAME = "yolov8n.pt"
YOLO_DEVICE = "cpu"
YOLO_WEIGHTS_PATH = Path("/kaggle/input/yolo-weights/yolov8n.pt")

DATA_DIR = Path("/kaggle/input/competitions/nfl-player-contact-detection")
OUTPUT_DIR = Path("/kaggle/working")

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Competition data path not found: {DATA_DIR}")

print(f"NOTEBOOK_VERSION: {NOTEBOOK_VERSION}")
print(f"NOTEBOOK_NAME: {NOTEBOOK_NAME}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"YOLO_RESEARCH_MODE: {YOLO_RESEARCH_MODE}")

## 2. Helper Functions

The helpers keep the probe small: parse contact IDs, map a 10 Hz label step to
a 59.94 Hz video frame, draw helmet boxes, and load YOLO either from one
attached weights path or from an internet-enabled research download.

In [ ]:
def parse_contact_id(df: pd.DataFrame) -> pd.DataFrame:
    """Split Kaggle contact_id into play, step, player1, and player2 fields."""
    out = df.copy()
    parts = out[ID_COL].astype(str).str.split("_", expand=True)
    out["game_play"] = parts[0] + "_" + parts[1]
    out["step"] = parts[2].astype("int16")
    out["nfl_player_id_1"] = parts[3].astype(str)
    out["nfl_player_id_2"] = parts[4].astype(str)
    out["contact_type"] = np.where(
        out["nfl_player_id_2"].eq("G"), "ground", "player_player"
    )
    return out


def step_to_video_frame(step: int) -> int:
    """Map a 10 Hz label step to the synchronized video frame index."""
    return int(round(SNAP_FRAME + step * (VIDEO_FPS / TRACKING_HZ)))


def read_video_frame(video_path: Path, frame_number: int):
    """Read one frame from an mp4 file with OpenCV."""
    import cv2

    capture = cv2.VideoCapture(str(video_path))
    capture.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
    ok, frame = capture.read()
    capture.release()
    if not ok:
        raise ValueError(f"Could not read frame {frame_number} from {video_path}")
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def draw_boxes(
    frame,
    boxes: pd.DataFrame,
    target_players: set[str] | None = None,
    title: str | None = None,
):
    """Draw helmet boxes and highlight contact-candidate players."""
    target_players = target_players or set()
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.imshow(frame)
    ax.axis("off")
    if title:
        ax.set_title(title)
    for row in boxes.itertuples(index=False):
        player_id = str(row.nfl_player_id)
        is_target = player_id in target_players
        rect = patches.Rectangle(
            (row.left, row.top),
            row.width,
            row.height,
            linewidth=2.6 if is_target else 0.8,
            edgecolor="red" if is_target else "yellow",
            facecolor="none",
            alpha=0.9,
        )
        ax.add_patch(rect)
        if is_target:
            ax.text(
                row.left,
                max(row.top - 4, 0),
                player_id,
                color="white",
                fontsize=8,
                bbox={"facecolor": "red", "alpha": 0.8, "pad": 1},
            )
    return fig


def internet_available(url: str = "https://pypi.org/simple/ultralytics/") -> bool:
    """Check whether Kaggle can reach PyPI before installing packages."""
    try:
        with urllib.request.urlopen(url, timeout=8) as response:
            return response.status < 500
    except Exception as exc:
        print(f"Internet/PyPI check failed: {exc}")
        return False


def ensure_ultralytics() -> bool:
    """Install ultralytics only for internet-enabled research runs."""
    if importlib.util.find_spec("ultralytics") is not None:
        print("ultralytics is already installed.")
        return True
    if not YOLO_RESEARCH_MODE:
        print("ultralytics is unavailable and research mode is disabled.")
        return False
    if not internet_available():
        print("Skipping ultralytics install: Kaggle cannot reach PyPI.")
        return False

    print("Installing ultralytics for research-mode YOLO probing...")
    try:
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--no-cache-dir",
            "ultralytics",
        ])
    except Exception as exc:
        print(f"Could not install ultralytics: {exc}")
        return False
    return importlib.util.find_spec("ultralytics") is not None


def load_yolo_model():
    """Load YOLO from one offline path or from the research download."""
    if not ensure_ultralytics():
        return None

    from ultralytics import YOLO

    if YOLO_WEIGHTS_PATH.exists():
        print(f"Loading attached YOLO weights: {YOLO_WEIGHTS_PATH}")
        model = YOLO(str(YOLO_WEIGHTS_PATH))
    elif YOLO_RESEARCH_MODE:
        print(f"Downloading/loading YOLO model for research: {YOLO_MODEL_NAME}")
        model = YOLO(YOLO_MODEL_NAME)
    else:
        print(f"No offline YOLO weights found: {YOLO_WEIGHTS_PATH}")
        return None

    model.to(YOLO_DEVICE)
    print(f"YOLO_DEVICE: {YOLO_DEVICE}")
    return model

## 3. Load Data and Pick Contact Examples

The probe only needs labels and baseline helmet boxes. We inspect one positive
player-player contact and one positive ground contact to cover both target
families.

In [ ]:
labels = pd.read_csv(DATA_DIR / "train_labels.csv")
labels = parse_contact_id(labels)
helmets = pd.read_csv(DATA_DIR / "train_baseline_helmets.csv")

positive_examples = (
    labels.loc[labels[TARGET].eq(1)]
    .groupby("contact_type", group_keys=False)
    .head(1)
    .copy()
)
positive_examples["video_frame"] = positive_examples["step"].map(step_to_video_frame)
display(positive_examples[[
    ID_COL, "game_play", "step", "video_frame", "contact_type",
    "nfl_player_id_1", "nfl_player_id_2", TARGET,
]])

print(f"labels: {labels.shape}")
print(f"helmets: {helmets.shape}")

## 4. Helmet Overlay Demo

The baseline helmet files already provide player assignments. Before adding
YOLO, we should understand whether the candidate players are visible and
whether the provided boxes line up with the contact frame.

In [ ]:
demo_row = positive_examples.iloc[0]
demo_view = "Sideline"
demo_frame = int(demo_row["video_frame"])
demo_video = helmets.loc[
    helmets["game_play"].eq(demo_row["game_play"])
    & helmets["view"].eq(demo_view),
    "video",
].iloc[0]
demo_boxes = helmets.loc[
    helmets["video"].eq(demo_video)
    & helmets["frame"].eq(demo_frame)
].copy()
target_players = {str(demo_row["nfl_player_id_1"])}
if demo_row["nfl_player_id_2"] != "G":
    target_players.add(str(demo_row["nfl_player_id_2"]))

print(f"video: {demo_video}")
print(f"frame: {demo_frame}")
print(f"boxes on frame: {len(demo_boxes)}")
frame = read_video_frame(DATA_DIR / "train" / demo_video, demo_frame)
fig = draw_boxes(
    frame,
    demo_boxes,
    target_players,
    title=f"{demo_row['contact_type']} contact candidate",
)
if fig is not None:
    output_path = OUTPUT_DIR / "yolo_probe_helmet_overlay.png"
    fig.savefig(output_path, bbox_inches="tight")
    plt.show()
    display(Image(filename=str(output_path)))

## 5. YOLO Frame Probe

YOLO is run on CPU to avoid Kaggle GPU/CUDA compatibility issues. In the latest
run, generic `yolov8n.pt` installed and downloaded successfully, but it detected
only a few objects on a crowded football frame. That makes YOLO useful for
qualitative exploration, while helmet boxes remain the more reliable
player-specific feature source.

In [ ]:
yolo_model = load_yolo_model()

if yolo_model is None:
    print("Skipping YOLO inference: no model is available.")
elif frame is None:
    print("Skipping YOLO inference: no readable demo frame.")
else:
    try:
        result = yolo_model.predict(
            frame,
            conf=YOLO_CONFIDENCE,
            device=YOLO_DEVICE,
            verbose=False,
        )[0]
    except Exception as exc:
        message = str(exc)
        if "CUDA" in message or "AcceleratorError" in type(exc).__name__:
            print("YOLO hit a CUDA accelerator error; retrying on CPU.")
            YOLO_DEVICE = "cpu"
            yolo_model.to(YOLO_DEVICE)
            result = yolo_model.predict(
                frame,
                conf=YOLO_CONFIDENCE,
                device=YOLO_DEVICE,
                verbose=False,
            )[0]
        else:
            raise

    detections = []
    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy.cpu().numpy()[0]
        cls_id = int(box.cls.cpu().numpy()[0])
        confidence = float(box.conf.cpu().numpy()[0])
        class_name = result.names.get(cls_id, str(cls_id))
        detections.append({
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
            "class_id": cls_id,
            "class_name": class_name,
            "confidence": confidence,
            "width": x2 - x1,
            "height": y2 - y1,
            "area": (x2 - x1) * (y2 - y1),
        })
    detections = pd.DataFrame(detections)
    print(f"YOLO_DEVICE used: {YOLO_DEVICE}")
    print(f"detections: {len(detections)}")
    display(detections.head(30))

    if len(detections):
        fig, ax = plt.subplots(figsize=(12, 7))
        ax.imshow(frame)
        ax.axis("off")
        ax.set_title("YOLO detections on contact frame")
        for row in detections.itertuples(index=False):
            rect = patches.Rectangle(
                (row.x1, row.y1),
                row.width,
                row.height,
                linewidth=1.6,
                edgecolor="lime",
                facecolor="none",
                alpha=0.9,
            )
            ax.add_patch(rect)
            ax.text(
                row.x1,
                max(row.y1 - 4, 0),
                f"{row.class_name} {row.confidence:.2f}",
                color="black",
                fontsize=8,
                bbox={"facecolor": "lime", "alpha": 0.75, "pad": 1},
            )
        output_path = OUTPUT_DIR / "yolo_probe_detections.png"
        fig.savefig(output_path, bbox_inches="tight")
        plt.show()
        display(Image(filename=str(output_path)))

## 6. Helmet-Derived Feature Probe

These compact features are cheap enough to consider for the tracking model:
player box visibility, box area, and player-pair pixel distance in each view.
They are not YOLO-dependent, so they can be used even if external detector
weights are unavailable.

In [ ]:
feature_sample = labels.sample(
    min(SAMPLE_ROWS, len(labels)),
    random_state=RANDOM_STATE,
).copy()
positives = labels.loc[labels[TARGET].eq(1)].sample(
    min(10_000, int(labels[TARGET].sum())),
    random_state=RANDOM_STATE,
)
feature_sample = pd.concat([feature_sample, positives], ignore_index=True)
feature_sample = feature_sample.drop_duplicates(ID_COL)
feature_sample["frame"] = feature_sample["step"].map(step_to_video_frame)
feature_sample["p1_id"] = feature_sample["nfl_player_id_1"].astype("int64")
feature_sample["p2_id"] = pd.to_numeric(
    feature_sample["nfl_player_id_2"], errors="coerce"
)

view = "Sideline"
view_helmets = helmets.loc[helmets["view"].eq(view)].copy()
box_cols = ["game_play", "frame", "nfl_player_id", "left", "width", "top", "height"]
p1_boxes = view_helmets[box_cols].rename(columns={
    "nfl_player_id": "p1_id",
    "left": "p1_left",
    "width": "p1_width",
    "top": "p1_top",
    "height": "p1_height",
})
p2_boxes = view_helmets[box_cols].rename(columns={
    "nfl_player_id": "p2_id",
    "left": "p2_left",
    "width": "p2_width",
    "top": "p2_top",
    "height": "p2_height",
})
features = feature_sample.merge(
    p1_boxes,
    on=["game_play", "frame", "p1_id"],
    how="left",
).merge(
    p2_boxes,
    on=["game_play", "frame", "p2_id"],
    how="left",
)
features["p1_box_area"] = features["p1_width"] * features["p1_height"]
features["p2_box_area"] = features["p2_width"] * features["p2_height"]
features["p1_visible"] = features["p1_box_area"].notna().astype("int8")
features["p2_visible"] = features["p2_box_area"].notna().astype("int8")
features["both_players_visible"] = (
    features["p1_visible"].eq(1) & features["p2_visible"].eq(1)
).astype("int8")
features["p1_center_x"] = features["p1_left"] + features["p1_width"] / 2
features["p1_center_y"] = features["p1_top"] + features["p1_height"] / 2
features["p2_center_x"] = features["p2_left"] + features["p2_width"] / 2
features["p2_center_y"] = features["p2_top"] + features["p2_height"] / 2
features["helmet_pair_distance_px"] = np.sqrt(
    (features["p1_center_x"] - features["p2_center_x"]) ** 2
    + (features["p1_center_y"] - features["p2_center_y"]) ** 2
)

summary_cols = [
    "p1_visible",
    "p2_visible",
    "both_players_visible",
    "p1_box_area",
    "p2_box_area",
    "helmet_pair_distance_px",
]
display(features.groupby(["contact_type", TARGET], observed=True)[summary_cols].mean())
display(features[summary_cols + ["contact_type", TARGET]].describe(include="all"))

## 7. Decision Notes

The current evidence favors helmet-derived video features before a full YOLO or
CNN pipeline. The baseline helmet boxes are player-assigned and dense enough for
visibility, box area, movement, and pixel-distance features. Generic YOLOv8 is
helpful for visual inspection, but on the sample frame it produced sparse and
noisy detections compared with the 22 helmet boxes.

Next production candidate: add Sideline/Endzone helmet visibility, box geometry,
helmet-pair pixel distance, and short-window/interpolated box features to the
tracking model.